In [11]:
!pip install -q transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
import os

In [12]:
if not os.path.exists('/content/NLPLabs-2024'):
    !git clone https://github.com/CRLala/NLPLabs-2024.git /content/NLPLabs-2024

!wget -q -O /content/train_ids.csv \
  "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv"

!wget -q -O /content/dev_ids.csv \
  "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv"

!wget -q -O /content/test.tsv \
  "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv"

In [13]:
import ast

df_pcl = pd.read_csv(
    '/content/NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv',
    sep='\t', header=None,
    names=['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']
)

df_pcl['par_id'] = pd.to_numeric(df_pcl['par_id'], errors='coerce')
df_pcl = df_pcl.dropna(subset=['par_id'])
df_pcl['par_id'] = df_pcl['par_id'].astype(int)

def load_split(path):
    df = pd.read_csv(path, header=0)
    df.columns = ['par_id', 'label']
    df['par_id'] = pd.to_numeric(df['par_id'], errors='coerce')

    def to_binary(s):
        vec = ast.literal_eval(s)
        return int(any(v == 1 for v in vec))

    df['label'] = df['label'].apply(to_binary)
    return df.dropna(subset=['par_id']).astype({'par_id': int})

train_ids = load_split('/content/train_ids.csv')
dev_ids   = load_split('/content/dev_ids.csv')

train_df = train_ids.merge(df_pcl[['par_id', 'text']], on='par_id')
dev_df   = dev_ids.merge(df_pcl[['par_id', 'text']], on='par_id')

test_df = pd.read_csv('/content/test.tsv', sep='\t', header=None,
                      names=['par_id', 'art_id', 'keyword', 'country_code', 'text'])

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print(f"Train PCL rate: {train_df['label'].mean():.2%}")
print(f"Dev PCL rate:   {dev_df['label'].mean():.2%}")

Train: 8375 | Dev: 2094 | Test: 3832
Train PCL rate: 9.48%
Dev PCL rate:   9.50%


In [14]:
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels=None):
        texts = [str(t) if t is not None and t == t else '' for t in texts]

        self.encodings = tokenizer(
            texts,
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = PCLDataset(train_df['text'].tolist(), train_df['label'].tolist())
dev_dataset   = PCLDataset(dev_df['text'].tolist(),   dev_df['label'].tolist())
test_dataset  = PCLDataset(test_df['text'].tolist())

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("Datasets ready.")
print(f"Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)} | Test batches: {len(test_loader)}")

Datasets ready.
Train batches: 524 | Dev batches: 66 | Test batches: 120


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

# weighted loss
n_neg = (train_df['label'] == 0).sum()
n_pos = (train_df['label'] == 1).sum()
weight = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=weight)
print(f"Class weights: [No PCL: 1.0, PCL: {n_neg/n_pos:.2f}]")

EPOCHS = 4
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

Using: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: [No PCL: 1.0, PCL: 9.55]


In [16]:
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
            preds   = (probs >= threshold).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, pos_label=1)
    return f1, all_preds, all_labels

best_f1    = 0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    f1, _, _ = evaluate(model, dev_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Dev F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_epoch = epoch + 1
        model.save_pretrained('/content/best_model')
        tokenizer.save_pretrained('/content/best_model')

print(f"\nBest Dev F1: {best_f1:.4f} at epoch {best_epoch}")

Epoch 1/4 | Loss: 0.6305 | Dev F1: 0.5069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/4 | Loss: 0.4820 | Dev F1: 0.5163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3/4 | Loss: 0.4014 | Dev F1: 0.5598


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4/4 | Loss: 0.2671 | Dev F1: 0.5820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best Dev F1: 0.5820 at epoch 4


In [17]:
# threshold tuning

from transformers import AutoModelForSequenceClassification
best_model = AutoModelForSequenceClassification.from_pretrained('/content/best_model')
best_model.to(device)
best_model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in dev_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(batch['labels'].numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

best_threshold = 0.5
best_thresh_f1 = 0

for thresh in np.arange(0.1, 0.9, 0.01):
    preds = (all_probs >= thresh).astype(int)
    f1    = f1_score(all_labels, preds, pos_label=1)
    if f1 > best_thresh_f1:
        best_thresh_f1 = f1
        best_threshold = thresh

print(f"Best threshold: {best_threshold:.2f} → Dev F1: {best_thresh_f1:.4f}")

preds = (all_probs >= best_threshold).astype(int)
print(classification_report(all_labels, preds, target_names=['No PCL', 'PCL']))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Best threshold: 0.55 → Dev F1: 0.5882
              precision    recall  f1-score   support

      No PCL       0.95      0.97      0.96      1895
         PCL       0.63      0.55      0.59       199

    accuracy                           0.93      2094
   macro avg       0.79      0.76      0.77      2094
weighted avg       0.92      0.93      0.92      2094



In [18]:
# dev.txt
dev_preds = (all_probs >= best_threshold).astype(int)
with open('/content/dev.txt', 'w') as f:
    for p in dev_preds:
        f.write(f"{p}\n")

# test.txt
best_model.eval()
test_probs = []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        test_probs.extend(probs)

test_preds = (np.array(test_probs) >= best_threshold).astype(int)
with open('/content/test.txt', 'w') as f:
    for p in test_preds:
        f.write(f"{p}\n")

print(f"dev.txt:  {len(dev_preds)} lines")
print(f"test.txt: {len(test_preds)} lines")
print(f"Test predicted PCL rate: {test_preds.mean():.2%}")

dev.txt:  2094 lines
test.txt: 3832 lines
Test predicted PCL rate: 7.05%


In [19]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
save_dir = '/content/drive/MyDrive/NLP_CW/BestModel'
os.makedirs(save_dir, exist_ok=True)

# Copy model files
shutil.copytree('/content/best_model', f'{save_dir}/model', dirs_exist_ok=True)

# Copy prediction files
shutil.copy('/content/dev.txt',  '/content/drive/MyDrive/NLP_CW/')
shutil.copy('/content/test.txt', '/content/drive/MyDrive/NLP_CW/')

print("All saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All saved to Drive.
